# Baseline Error Analysis Notebook

这个 notebook 面向 `tools/relation_test_net.py` 产出的离线评估结果，重点回答：

- baseline 最容易混淆哪些 predicate / 词汇
- 这些错误最常发生在哪些 subject-object 对上
- 错误是否与长尾分布、稀有 triplet、未见 triplet 有关
- 哪些错误是高置信度错例，适合做案例分析

默认读取：

- `RESULT_DIR/eval_results.pytorch`
- `RESULT_DIR/visual_info.json`

设计原则：

- 不依赖重新跑模型
- 不依赖日志字符串
- 直接复用仓库里的 `BoxList` 字段和 VG 词表
- 对 `predcls` / `sgcls` / `sgdet` 都能工作，并清楚说明 `sgdet` 下的匹配近似规则


## 1. Environment Setup

In [ ]:
import json
import math
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from maskrcnn_benchmark.config import cfg
from maskrcnn_benchmark.config.paths_catalog import DatasetCatalog
from maskrcnn_benchmark.data.datasets.visual_genome import VGDataset

plt.style.use('default')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 200)

print('Repo root:', REPO_ROOT)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## 2. Configuration

In [ ]:
RESULT_DIR = Path('/workspace/ccloud/sf/SDSGG/output/baseline/inference/VG_stanford_filtered_with_attribute_test')
TOPK_CONFUSIONS = 30
TOPK_PREDICATES = 20
TOPK_PAIR_PATTERNS = 30
MIN_GT_COUNT = 5
ENABLE_TRAIN_COMPARISON = True
TRAIN_SPLIT_NAME = 'VG_stanford_filtered_with_attribute_train'
CONFIG_FILE = REPO_ROOT / 'configs/e2e_relation_X_101_32_8_FPN_1x_total.yaml'
TRAIN_DATASET_NAME_OVERRIDE = None
SGDET_IOU_THRESHOLD = 0.5
CASE_SAMPLE_LIMIT = 6
CASE_FILTER_GT_PREDICATE = None
CASE_FILTER_PRED_PREDICATE = None

EVAL_RESULTS_PATH = RESULT_DIR / 'eval_results.pytorch'
VISUAL_INFO_PATH = RESULT_DIR / 'visual_info.json'

print('RESULT_DIR =', RESULT_DIR)
print('EVAL_RESULTS_PATH exists =', EVAL_RESULTS_PATH.exists())
print('VISUAL_INFO_PATH exists =', VISUAL_INFO_PATH.exists())
print('ENABLE_TRAIN_COMPARISON =', ENABLE_TRAIN_COMPARISON)


## 3. Helpers

In [ ]:
def assert_exists(path: Path, description: str):
    if not path.exists():
        raise FileNotFoundError(f'{description} not found: {path}')


def normalize_name(name):
    return ' '.join(str(name).strip().lower().split())


def load_result_artifacts(result_dir: Path):
    eval_results_path = result_dir / 'eval_results.pytorch'
    visual_info_path = result_dir / 'visual_info.json'
    assert_exists(eval_results_path, 'eval_results.pytorch')
    assert_exists(visual_info_path, 'visual_info.json')
    payload = torch.load(eval_results_path, map_location=torch.device('cpu'))
    with open(visual_info_path, 'r', encoding='utf-8') as handle:
        visual_info = json.load(handle)
    if 'groundtruths' not in payload or 'predictions' not in payload:
        raise KeyError('eval_results.pytorch must contain groundtruths and predictions')
    return payload['groundtruths'], payload['predictions'], visual_info


def infer_mode(groundtruths, predictions):
    gt0 = groundtruths[0]
    pred0 = predictions[0]
    gt_boxes = gt0.convert('xyxy').bbox.detach().cpu().numpy()
    gt_labels = gt0.get_field('labels').detach().cpu().numpy()
    pred_boxes = pred0.convert('xyxy').bbox.detach().cpu().numpy()
    pred_labels = pred0.get_field('pred_labels').detach().cpu().numpy()
    same_count = len(gt_boxes) == len(pred_boxes)
    same_boxes = same_count and np.allclose(gt_boxes, pred_boxes)
    same_labels = same_count and np.array_equal(gt_labels, pred_labels)
    if same_boxes and same_labels:
        return 'predcls'
    if same_boxes:
        return 'sgcls'
    return 'sgdet'


def box_iou_xyxy(box_a, box_b):
    xa1, ya1, xa2, ya2 = [float(v) for v in box_a]
    xb1, yb1, xb2, yb2 = [float(v) for v in box_b]
    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)
    inter_w = max(0.0, inter_x2 - inter_x1 + 1.0)
    inter_h = max(0.0, inter_y2 - inter_y1 + 1.0)
    inter = inter_w * inter_h
    area_a = max(0.0, xa2 - xa1 + 1.0) * max(0.0, ya2 - ya1 + 1.0)
    area_b = max(0.0, xb2 - xb1 + 1.0) * max(0.0, yb2 - yb1 + 1.0)
    denom = area_a + area_b - inter
    return 0.0 if denom <= 0 else inter / denom


def safe_name(vocab, idx):
    idx = int(idx)
    if 0 <= idx < len(vocab):
        return vocab[idx]
    return f'<unk:{idx}>'


def _infer_dataset_name(cfg_obj, split_name, override=None):
    if override:
        return override
    dataset_names = getattr(cfg_obj.DATASETS, split_name.upper(), ())
    if dataset_names:
        return dataset_names[0]
    return f'VG_stanford_filtered_with_attribute_{split_name}'


def build_vg_dataset(split_name='train', config_file=None, dataset_name_override=None):
    cfg_local = cfg.clone()
    if config_file and Path(config_file).exists():
        cfg_local.merge_from_file(str(config_file))
    cfg_local.freeze()
    dataset_name = _infer_dataset_name(cfg_local, split_name, dataset_name_override)
    dataset_info = DatasetCatalog.get(dataset_name, cfg_local)
    if dataset_info['factory'] != 'VGDataset':
        raise ValueError(f'Expected VGDataset, got {dataset_info["factory"]}')
    dataset_args = dict(dataset_info['args'])
    dataset_args.pop('capgraphs_file', None)
    dataset_args['transforms'] = None
    return VGDataset(**dataset_args), dataset_name


def collect_dataset_triplets(dataset):
    triplets = []
    predicate_counter = Counter()
    for image_index in range(len(dataset)):
        gt = dataset.get_groundtruth(image_index, evaluation=True, flip_img=False)
        rels = gt.get_field('relation_tuple').long().detach().cpu().numpy()
        labels = gt.get_field('labels').long().detach().cpu().numpy()
        for sub_idx, obj_idx, pred_idx in rels:
            sub_cls = int(labels[sub_idx])
            obj_cls = int(labels[obj_idx])
            pred_idx = int(pred_idx)
            triplets.append((sub_cls, pred_idx, obj_cls))
            predicate_counter[pred_idx] += 1
    return set(triplets), predicate_counter


def top1_predicates(pred_rel_scores):
    rel_scores = pred_rel_scores.detach().cpu().numpy()
    if rel_scores.shape[1] <= 1:
        pred_labels = np.zeros(rel_scores.shape[0], dtype=np.int64)
        pred_scores = np.zeros(rel_scores.shape[0], dtype=np.float32)
    else:
        pred_labels = rel_scores[:, 1:].argmax(1) + 1
        pred_scores = rel_scores[:, 1:].max(1)
    return pred_labels.astype(np.int64), pred_scores.astype(np.float32), rel_scores


def analyze_predictions(groundtruths, predictions, visual_info, predicate_names, object_names, mode, iou_threshold=0.5):
    rows = []
    raw_pred_rows = []
    visual_lookup = {idx: item for idx, item in enumerate(visual_info)}

    for image_id, (gt, pred) in enumerate(zip(groundtruths, predictions)):
        img_meta = visual_lookup.get(image_id, {})
        img_file = img_meta.get('img_file')

        gt_boxes = gt.convert('xyxy').bbox.detach().cpu().numpy()
        gt_labels = gt.get_field('labels').long().detach().cpu().numpy()
        gt_rels = gt.get_field('relation_tuple').long().detach().cpu().numpy()

        pred_boxes = pred.convert('xyxy').bbox.detach().cpu().numpy()
        pred_labels = pred.get_field('pred_labels').long().detach().cpu().numpy()
        pred_obj_scores = pred.get_field('pred_scores').detach().cpu().numpy()
        pred_rel_inds = pred.get_field('rel_pair_idxs').long().detach().cpu().numpy()
        pred_rel_top1, pred_rel_top1_scores, pred_rel_all_scores = top1_predicates(pred.get_field('pred_rel_scores'))

        for pred_rel_idx, (pair, pred_predicate, pred_score) in enumerate(zip(pred_rel_inds, pred_rel_top1, pred_rel_top1_scores)):
            sub_idx, obj_idx = [int(v) for v in pair]
            raw_pred_rows.append({
                'image_id': image_id,
                'img_file': img_file,
                'pred_rel_index': pred_rel_idx,
                'pred_predicate': int(pred_predicate),
                'pred_predicate_name': safe_name(predicate_names, pred_predicate),
                'pred_score': float(pred_score),
                'pred_sub_class': int(pred_labels[sub_idx]),
                'pred_obj_class': int(pred_labels[obj_idx]),
                'pred_sub_name': safe_name(object_names, pred_labels[sub_idx]),
                'pred_obj_name': safe_name(object_names, pred_labels[obj_idx]),
            })

        for gt_rel_index, (gt_sub_idx, gt_obj_idx, gt_predicate) in enumerate(gt_rels):
            gt_sub_idx = int(gt_sub_idx)
            gt_obj_idx = int(gt_obj_idx)
            gt_predicate = int(gt_predicate)

            gt_sub_class = int(gt_labels[gt_sub_idx])
            gt_obj_class = int(gt_labels[gt_obj_idx])
            gt_sub_box = gt_boxes[gt_sub_idx].tolist()
            gt_obj_box = gt_boxes[gt_obj_idx].tolist()

            matched_pred_rel_index = None
            matched_predicate = None
            matched_pred_score = None
            matched_sub_box = None
            matched_obj_box = None
            matching_rule = 'none'
            best_key = None

            for pred_rel_index, (pair, pred_predicate, pred_score) in enumerate(zip(pred_rel_inds, pred_rel_top1, pred_rel_top1_scores)):
                pred_sub_idx, pred_obj_idx = [int(v) for v in pair]
                pred_sub_class = int(pred_labels[pred_sub_idx])
                pred_obj_class = int(pred_labels[pred_obj_idx])

                if mode in {'predcls', 'sgcls'}:
                    if pred_sub_idx != gt_sub_idx or pred_obj_idx != gt_obj_idx:
                        continue
                    candidate_quality = 2.0
                    candidate_rule = 'same_pair_index'
                else:
                    if pred_sub_class != gt_sub_class or pred_obj_class != gt_obj_class:
                        continue
                    sub_iou = box_iou_xyxy(gt_boxes[gt_sub_idx], pred_boxes[pred_sub_idx])
                    obj_iou = box_iou_xyxy(gt_boxes[gt_obj_idx], pred_boxes[pred_obj_idx])
                    if sub_iou < iou_threshold or obj_iou < iou_threshold:
                        continue
                    candidate_quality = min(sub_iou, obj_iou)
                    candidate_rule = 'class_and_iou_match'

                candidate_key = (float(pred_score), float(candidate_quality), -pred_rel_index)
                if best_key is None or candidate_key > best_key:
                    best_key = candidate_key
                    matched_pred_rel_index = pred_rel_index
                    matched_predicate = int(pred_predicate)
                    matched_pred_score = float(pred_score)
                    matched_sub_box = pred_boxes[pred_sub_idx].tolist()
                    matched_obj_box = pred_boxes[pred_obj_idx].tolist()
                    matching_rule = candidate_rule

            is_matched = matched_predicate is not None
            is_correct = is_matched and matched_predicate == gt_predicate
            rows.append({
                'image_id': image_id,
                'img_file': img_file,
                'mode': mode,
                'gt_rel_index': gt_rel_index,
                'gt_sub_idx': gt_sub_idx,
                'gt_obj_idx': gt_obj_idx,
                'gt_predicate': gt_predicate,
                'gt_predicate_name': safe_name(predicate_names, gt_predicate),
                'gt_sub_class': gt_sub_class,
                'gt_obj_class': gt_obj_class,
                'gt_sub_name': safe_name(object_names, gt_sub_class),
                'gt_obj_name': safe_name(object_names, gt_obj_class),
                'gt_sub_box': gt_sub_box,
                'gt_obj_box': gt_obj_box,
                'matched_pred_rel_index': matched_pred_rel_index,
                'pred_predicate': matched_predicate,
                'pred_predicate_name': None if matched_predicate is None else safe_name(predicate_names, matched_predicate),
                'pred_score': matched_pred_score,
                'pred_sub_box': matched_sub_box,
                'pred_obj_box': matched_obj_box,
                'matching_rule': matching_rule,
                'is_matched': bool(is_matched),
                'is_correct': bool(is_correct),
                'is_error': bool(not is_correct),
                'error_type': 'missed_pair' if not is_matched else ('wrong_predicate' if not is_correct else 'correct'),
            })

    return pd.DataFrame(rows), pd.DataFrame(raw_pred_rows)


def summarize_results(match_df, raw_pred_df, predicate_names):
    pred_indices = [idx for idx, name in enumerate(predicate_names) if normalize_name(name) != '__background__']
    gt_counts = match_df.groupby('gt_predicate').size().rename('gt_count') if not match_df.empty else pd.Series(dtype='int64')
    hit_counts = match_df[match_df['is_correct']].groupby('gt_predicate').size().rename('hit_count') if not match_df.empty else pd.Series(dtype='int64')
    matched_counts = match_df[match_df['is_matched']].groupby('gt_predicate').size().rename('matched_count') if not match_df.empty else pd.Series(dtype='int64')
    raw_pred_counts = raw_pred_df.groupby('pred_predicate').size().rename('raw_pred_count') if not raw_pred_df.empty else pd.Series(dtype='int64')

    summary = pd.DataFrame(index=pred_indices)
    summary.index.name = 'predicate_idx'
    summary['predicate_name'] = [safe_name(predicate_names, idx) for idx in summary.index]
    summary = summary.join(gt_counts, how='left')
    summary = summary.join(hit_counts, how='left')
    summary = summary.join(matched_counts, how='left')
    summary = summary.join(raw_pred_counts, how='left')
    summary = summary.fillna(0)
    for col in ['gt_count', 'hit_count', 'matched_count', 'raw_pred_count']:
        summary[col] = summary[col].astype(int)
    summary['approx_recall'] = summary['hit_count'] / summary['gt_count'].replace(0, np.nan)
    summary['approx_precision'] = summary['hit_count'] / summary['raw_pred_count'].replace(0, np.nan)
    summary['match_rate'] = summary['matched_count'] / summary['gt_count'].replace(0, np.nan)
    summary = summary.sort_values(['gt_count', 'approx_recall', 'predicate_name'], ascending=[False, True, True])
    return summary.reset_index()


def build_confusion_table(match_df):
    if match_df.empty:
        return pd.DataFrame()
    wrong_df = match_df[(match_df['is_matched']) & (~match_df['is_correct'])].copy()
    if wrong_df.empty:
        return pd.DataFrame()
    table = (
        wrong_df.groupby(['gt_predicate', 'gt_predicate_name', 'pred_predicate', 'pred_predicate_name'])
        .agg(
            count=('image_id', 'size'),
            mean_score=('pred_score', 'mean'),
            median_score=('pred_score', 'median'),
            unique_images=('image_id', 'nunique'),
        )
        .reset_index()
        .sort_values(['count', 'mean_score'], ascending=[False, False])
    )
    return table


def build_pair_pattern_table(match_df):
    if match_df.empty:
        return pd.DataFrame()
    wrong_df = match_df[(match_df['is_matched']) & (~match_df['is_correct'])].copy()
    if wrong_df.empty:
        return pd.DataFrame()
    pair_table = (
        wrong_df.groupby([
            'gt_sub_name', 'gt_obj_name', 'gt_predicate_name', 'pred_predicate_name'
        ])
        .agg(
            count=('image_id', 'size'),
            mean_score=('pred_score', 'mean'),
            unique_images=('image_id', 'nunique'),
        )
        .reset_index()
        .sort_values(['count', 'mean_score'], ascending=[False, False])
    )
    return pair_table


def assign_frequency_bins(summary_df):
    valid = summary_df[summary_df['gt_count'] > 0].copy()
    if valid.empty:
        summary_df = summary_df.copy()
        summary_df['freq_bin'] = 'empty'
        return summary_df
    quantiles = valid['gt_count'].quantile([1/3, 2/3]).to_dict()
    q1 = float(quantiles[1/3])
    q2 = float(quantiles[2/3])

    def to_bin(count):
        if count <= 0:
            return 'empty'
        if count <= q1:
            return 'tail'
        if count <= q2:
            return 'medium'
        return 'head'

    summary_df = summary_df.copy()
    summary_df['freq_bin'] = summary_df['gt_count'].map(to_bin)
    return summary_df


def build_bin_summary(summary_with_bins):
    valid = summary_with_bins[summary_with_bins['gt_count'] > 0].copy()
    if valid.empty:
        return pd.DataFrame()
    out = (
        valid.groupby('freq_bin')
        .agg(
            predicates=('predicate_name', 'size'),
            total_gt=('gt_count', 'sum'),
            total_hit=('hit_count', 'sum'),
            macro_recall=('approx_recall', 'mean'),
            macro_precision=('approx_precision', 'mean'),
        )
        .reset_index()
    )
    out['micro_recall'] = out['total_hit'] / out['total_gt'].replace(0, np.nan)
    return out.sort_values('freq_bin')


def add_train_comparison(match_df, summary_df, predicate_names, object_names, config_file=None, train_split_name='train', dataset_name_override=None):
    try:
        train_dataset, resolved_name = build_vg_dataset(train_split_name, config_file=config_file, dataset_name_override=dataset_name_override)
        train_triplets, train_predicate_counter = collect_dataset_triplets(train_dataset)
    except Exception as exc:
        print('Train comparison disabled:', exc)
        return match_df, summary_df, None, None

    match_df = match_df.copy()
    match_df['triplet_key'] = list(zip(match_df['gt_sub_class'], match_df['gt_predicate'], match_df['gt_obj_class']))
    match_df['seen_in_train'] = match_df['triplet_key'].isin(train_triplets)

    train_pred_df = pd.DataFrame([
        {
            'predicate_idx': idx,
            'train_gt_count': int(train_predicate_counter.get(idx, 0)),
            'predicate_name': safe_name(predicate_names, idx),
        }
        for idx in summary_df['predicate_idx'].tolist()
    ])
    summary_df = summary_df.merge(train_pred_df[['predicate_idx', 'train_gt_count']], on='predicate_idx', how='left')
    summary_df['train_gt_count'] = summary_df['train_gt_count'].fillna(0).astype(int)

    triplet_summary = (
        match_df.groupby('seen_in_train')
        .agg(
            gt_relations=('image_id', 'size'),
            hits=('is_correct', 'sum'),
            matched=('is_matched', 'sum'),
            unique_triplets=('triplet_key', 'nunique'),
            mean_score=('pred_score', 'mean'),
        )
        .reset_index()
    )
    triplet_summary['approx_recall'] = triplet_summary['hits'] / triplet_summary['gt_relations'].replace(0, np.nan)
    triplet_summary['match_rate'] = triplet_summary['matched'] / triplet_summary['gt_relations'].replace(0, np.nan)
    triplet_summary['seen_in_train'] = triplet_summary['seen_in_train'].map({True: 'seen', False: 'unseen'})
    return match_df, summary_df, triplet_summary, resolved_name


def build_score_bucket_table(match_df, bins=None):
    if match_df.empty:
        return pd.DataFrame()
    score_df = match_df[match_df['is_matched']].copy()
    if score_df.empty:
        return pd.DataFrame()
    if bins is None:
        bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.000001]
    score_df['score_bucket'] = pd.cut(score_df['pred_score'], bins=bins, include_lowest=True)
    bucket_table = (
        score_df.groupby('score_bucket')
        .agg(
            matched_relations=('image_id', 'size'),
            correct=('is_correct', 'sum'),
            wrong=('is_error', 'sum'),
        )
        .reset_index()
    )
    bucket_table['error_rate'] = bucket_table['wrong'] / bucket_table['matched_relations'].replace(0, np.nan)
    bucket_table['accuracy'] = bucket_table['correct'] / bucket_table['matched_relations'].replace(0, np.nan)
    bucket_table['score_bucket'] = bucket_table['score_bucket'].astype(str)
    return bucket_table


def build_high_confidence_error_table(match_df):
    if match_df.empty:
        return pd.DataFrame()
    wrong_df = match_df[(match_df['is_matched']) & (~match_df['is_correct'])].copy()
    if wrong_df.empty:
        return pd.DataFrame()
    return wrong_df.sort_values(['pred_score', 'image_id'], ascending=[False, True])


def plot_frequency_distribution(summary_df):
    valid = summary_df[summary_df['gt_count'] > 0].copy()
    if valid.empty:
        print('No predicate counts available for plotting.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].hist(valid['gt_count'], bins=min(30, max(5, len(valid))), color='#4C72B0', alpha=0.85)
    axes[0].set_title('GT Predicate Count Distribution')
    axes[0].set_xlabel('GT count')
    axes[0].set_ylabel('Number of predicates')

    plot_df = valid.sort_values('gt_count', ascending=False).head(TOPK_PREDICATES)
    axes[1].barh(plot_df['predicate_name'][::-1], plot_df['gt_count'][::-1], color='#55A868')
    axes[1].set_title(f'Top {min(TOPK_PREDICATES, len(plot_df))} Frequent Predicates')
    axes[1].set_xlabel('GT count')
    plt.tight_layout()


def show_case(row, title=None):
    img_file = row.get('img_file')
    if not img_file or not Path(img_file).exists():
        print('Image not available for row:', row.get('image_id'))
        return

    image = Image.open(img_file).convert('RGB')
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(image)
    ax.axis('off')

    def add_box(box, color, label):
        if box is None:
            return
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, max(0, y1 - 4), label, color='white', fontsize=10, bbox=dict(facecolor=color, alpha=0.75, pad=2))

    add_box(row.get('gt_sub_box'), '#2E86DE', f"GT SUB: {row['gt_sub_name']}")
    add_box(row.get('gt_obj_box'), '#E67E22', f"GT OBJ: {row['gt_obj_name']}")
    if row.get('pred_sub_box') is not None:
        add_box(row.get('pred_sub_box'), '#16A085', 'PRED SUB')
    if row.get('pred_obj_box') is not None:
        add_box(row.get('pred_obj_box'), '#C0392B', 'PRED OBJ')

    full_title = title or (
        f"image_id={row['image_id']} | {row['gt_sub_name']} - {row['gt_predicate_name']} - {row['gt_obj_name']}"
        f"\nPred: {row.get('pred_predicate_name')} | score={row.get('pred_score')} | match={row.get('matching_rule')}"
    )
    ax.set_title(full_title)
    plt.tight_layout()


def filter_cases(match_df, gt_predicate_name=None, pred_predicate_name=None, only_errors=True):
    case_df = match_df.copy()
    if only_errors:
        case_df = case_df[case_df['is_error']]
    if gt_predicate_name is not None:
        case_df = case_df[case_df['gt_predicate_name'] == gt_predicate_name]
    if pred_predicate_name is not None:
        case_df = case_df[case_df['pred_predicate_name'] == pred_predicate_name]
    return case_df.sort_values(['pred_score', 'image_id'], ascending=[False, True])


## 4. Load Results and Sanity Check

In [ ]:
groundtruths, predictions, visual_info = load_result_artifacts(RESULT_DIR)
mode = infer_mode(groundtruths, predictions)

first_gt = groundtruths[0]
first_pred = predictions[0]
num_predicates = int(first_pred.get_field('pred_rel_scores').shape[1])
num_objects = int(max(first_gt.get_field('labels').max().item(), first_pred.get_field('pred_labels').max().item()) + 1)

print('Loaded images:', len(groundtruths))
print('Loaded predictions:', len(predictions))
print('Loaded visual entries:', len(visual_info))
print('Inferred mode:', mode)
print('Predicate classes from pred_rel_scores:', num_predicates)
print('Max observed object class index + 1:', num_objects)
print('Prediction extra fields:', sorted(list(first_pred.extra_fields.keys())))
print('Groundtruth extra fields:', sorted(list(first_gt.extra_fields.keys())))


## 5. Build Vocabulary and Per-Relation Analysis Table

In [ ]:
try:
    vg_train_dataset, resolved_train_name = build_vg_dataset('train', config_file=CONFIG_FILE, dataset_name_override=TRAIN_DATASET_NAME_OVERRIDE)
    predicate_names = list(vg_train_dataset.ind_to_predicates)
    object_names = list(vg_train_dataset.ind_to_classes)
    print('Vocabulary source dataset:', resolved_train_name)
except Exception as exc:
    print('Failed to build VG dataset for vocabulary, fallback to synthetic names:', exc)
    predicate_names = [f'predicate_{i}' for i in range(num_predicates)]
    object_names = [f'object_{i}' for i in range(num_objects)]

if len(predicate_names) < num_predicates:
    predicate_names = predicate_names + [f'predicate_{i}' for i in range(len(predicate_names), num_predicates)]
if len(object_names) < num_objects:
    object_names = object_names + [f'object_{i}' for i in range(len(object_names), num_objects)]

match_df, raw_pred_df = analyze_predictions(
    groundtruths=groundtruths,
    predictions=predictions,
    visual_info=visual_info,
    predicate_names=predicate_names,
    object_names=object_names,
    mode=mode,
    iou_threshold=SGDET_IOU_THRESHOLD,
)

print('Per-GT relation rows:', len(match_df))
print('Raw predicted relation rows:', len(raw_pred_df))
display(match_df.head(3))


## 6. Global Predicate Performance

In [ ]:
predicate_summary = summarize_results(match_df, raw_pred_df, predicate_names)
predicate_summary = assign_frequency_bins(predicate_summary)

display(predicate_summary.head(TOPK_PREDICATES))
print('Worst predicates by approximate recall (filtered by MIN_GT_COUNT):')
display(
    predicate_summary[predicate_summary['gt_count'] >= MIN_GT_COUNT]
    .sort_values(['approx_recall', 'gt_count', 'predicate_name'], ascending=[True, False, True])
    .head(TOPK_PREDICATES)
)

bin_summary = build_bin_summary(predicate_summary)
display(bin_summary)
plot_frequency_distribution(predicate_summary)


## 7. Predicate Confusions

In [ ]:
confusion_table = build_confusion_table(match_df)
if confusion_table.empty:
    print('No matched wrong-predicate confusions found.')
else:
    display(confusion_table.head(TOPK_CONFUSIONS))

    confusion_matrix_df = (
        confusion_table.pivot_table(
            index='gt_predicate_name',
            columns='pred_predicate_name',
            values='count',
            fill_value=0,
            aggfunc='sum',
        )
    )

    top_gt = (
        predicate_summary[predicate_summary['gt_count'] >= MIN_GT_COUNT]
        .sort_values('gt_count', ascending=False)
        .head(min(TOPK_PREDICATES, len(predicate_summary)))['predicate_name']
        .tolist()
    )
    if top_gt:
        matrix_to_plot = confusion_matrix_df.reindex(index=top_gt, columns=top_gt, fill_value=0)
        plt.figure(figsize=(10, 8))
        plt.imshow(matrix_to_plot.values, cmap='Reds')
        plt.xticks(range(len(matrix_to_plot.columns)), matrix_to_plot.columns, rotation=90)
        plt.yticks(range(len(matrix_to_plot.index)), matrix_to_plot.index)
        plt.title('Predicate Confusion Matrix (Top Frequent GT Predicates)')
        plt.colorbar()
        plt.tight_layout()


## 8. Subject-Object Pair Error Patterns

In [ ]:
pair_pattern_table = build_pair_pattern_table(match_df)
if pair_pattern_table.empty:
    print('No subject-object specific wrong-predicate patterns found.')
else:
    display(pair_pattern_table.head(TOPK_PAIR_PATTERNS))

    pair_rollup = (
        pair_pattern_table.groupby(['gt_sub_name', 'gt_obj_name'])
        .agg(total_errors=('count', 'sum'), dominant_confusions=('gt_predicate_name', 'size'))
        .reset_index()
        .sort_values(['total_errors', 'dominant_confusions'], ascending=[False, False])
    )
    print('Most error-prone subject-object pairs:')
    display(pair_rollup.head(TOPK_PAIR_PATTERNS))


## 9. Train Comparison and Seen/Unseen Triplets

In [ ]:
train_triplet_summary = None
resolved_train_dataset_name = None
if ENABLE_TRAIN_COMPARISON:
    match_df, predicate_summary, train_triplet_summary, resolved_train_dataset_name = add_train_comparison(
        match_df=match_df,
        summary_df=predicate_summary,
        predicate_names=predicate_names,
        object_names=object_names,
        config_file=CONFIG_FILE,
        train_split_name='train',
        dataset_name_override=TRAIN_SPLIT_NAME,
    )

if train_triplet_summary is None:
    print('Train comparison unavailable. Notebook keeps running with test-only analysis.')
else:
    print('Resolved train dataset for comparison:', resolved_train_dataset_name)
    display(train_triplet_summary)
    print('Predicates with strongest train-test imbalance signals:')
    display(
        predicate_summary[predicate_summary['gt_count'] >= MIN_GT_COUNT]
        .sort_values(['train_gt_count', 'approx_recall'], ascending=[True, True])
        .head(TOPK_PREDICATES)
    )


## 10. Score Calibration and High-Confidence Errors

In [ ]:
score_bucket_table = build_score_bucket_table(match_df)
display(score_bucket_table)

if not score_bucket_table.empty:
    plt.figure(figsize=(8, 4))
    plt.plot(score_bucket_table['score_bucket'], score_bucket_table['error_rate'], marker='o', color='#C44E52')
    plt.xticks(rotation=30)
    plt.ylabel('Error rate')
    plt.xlabel('Score bucket')
    plt.title('Matched Relation Error Rate by Confidence Bucket')
    plt.tight_layout()

high_conf_error_table = build_high_confidence_error_table(match_df)
display(high_conf_error_table.head(TOPK_CONFUSIONS))


## 11. Visual Case Study

In [ ]:
case_df = filter_cases(
    match_df,
    gt_predicate_name=CASE_FILTER_GT_PREDICATE,
    pred_predicate_name=CASE_FILTER_PRED_PREDICATE,
    only_errors=True,
)

display(case_df.head(CASE_SAMPLE_LIMIT))
for _, row in case_df.head(CASE_SAMPLE_LIMIT).iterrows():
    show_case(row)
    plt.show()


## Notes on Interpretation

- `approx_recall` 和 `approx_precision` 是分析型指标，目的是帮助定位错误，不等同于官方 `R@K / mR@K`。
- 对 `predcls` / `sgcls`，这里的混淆归因接近“固定 pair 的 predicate 分类错误”。
- 对 `sgdet`，这里先要求 subject / object 类别一致，再要求框 IoU 满足阈值，然后在候选关系里选择得分最高者作为归因对象。
- 因此 `sgdet` 下的混淆表更适合做误差分析，不应直接当成官方评测替代品。
- 若 train 对照加载失败，notebook 会自动降级为 test-only 分析，不阻塞主要结论。
